# Tutorial - Virtual Screening

First, let's verify that `Boltz-tools` is installed correctly by running:

In [ ]:
!boltz-tools --version

If the installation was successful, this command will display the installed version number of `Boltz-tools`.

If you encounter any errors, refer to the installation guide in the `README.md` file to troubleshoot and complete the setup.

> **Note:**  
> - When running these commands in a regular command-line interface (CLI), remove the `!` at the beginning of the command.
> - If you're copying commands from a Jupyter Notebook that use backslashes (`\`) to split long commands across multiple lines, remove the backslashes and write the command on a single line in the CLI.

## Command and Input Files

The `screen` subcommand in `Boltz-tools` is used to perform virtual screening against a library of compounds. To use it effectively, it’s important to understand both the command structure and the required input files.

### 1. Command 

To understand how to use the screening workflow, start by viewing the command help:

In [ ]:
!boltz-tools screen --help

### Key Arguments and Options

| **Flag/Argument** | **Description** |
|-------------------|-----------------|
| `-w, --wrk_dir` | Working directory. Defaults to the current directory if not specified. |
| `-s, --system_path` | Path to the **system YAML file**, which defines the system (e.g., protein, receptor, or other setup details). |
| `-b, --boltz_options_path` | Path to the **Boltz options YAML file** specifying screening settings and parameters. |
| `-v, --variable` | Comma-separated nested path to update variables in the system YAML (e.g., `sequences,1,ligand,smiles`). |
| `-c, --variable_csv` | Path to a **CSV file** containing variables (e.g., ligands). Requires `--col_variable` and `--col_id`. |
| `--col_variable` | Column in the **CSV file** containing the variable, such as `SMILES` or `CCD` identifiers. |
| `--col_id` | Column in the **CSV file** containing the compound IDs. |
| `-s, --variable_sdf` | Path to an **SDF file** containing variables. Requires `--property_id`. |
| `--property_id` | Property in the **SDF file** used for compound IDs. |
| `--generate_conformers` | Optionally generate `2D` or `3D` conformers for input compounds (**works only for SMILES input**). |
| `--merge_data` | Optional: Merge extra data fields from input CSV or SDF into the output. |
| `-d, --debug` | Enables debug logging for troubleshooting. |


### Required Input Files

To run `boltz-tools screen`, you need the following files:

- A **system file** (`.yaml`) describing the biological system and initial settings  
- An **options file** (`.yaml`) specifying screening parameters and workflow settings  
- **Either** a **screening file** in **SDF** (`.sdf`) **or** **CSV** (`.csv`) format containing the ligands to be screened  

Let's go through the imput files:


#### System File

A minimal `system.yaml` file might look like this:

In [ ]:
!cat ../examples/system_screen.yaml

In this setup, the ligand does not yet have a SMILES string.
To specify the SMILES dynamically during screening, use the `-v` option in the command:
 - `-v sequences,1,ligand,smiles` tells `Boltz-tool`s to insert or update the `SMILES` value for the ligand entry.
 - If the system file already contained a `SMILES` value, this command would overwrite it with the values from the screening file (`.csv` or `.sdf`).

In Boltz-tools, the `-v` option allows you to **update any variable** in the system YAML dynamically, not just ligand SMILES.  

For example, you could modify:  
- Ligand SMILES  
- Protein sequences  
- Any other nested variable defined in the system  

The system file itself is also flexible: **any valid system YAML** that can be uploaded to Boltz-tools can be used here.  
This allows you to screen different ligands, protein variants, or combinations thereof, without modifying the original system file manually.  


#### Options File

The **options file** allows you to customize the behavior of `Boltz` during a screening run.  
To see the full list of available settings, you can run:

In [ ]:
!boltz predict --help

This will display all the command-line options that can also be set or adjusted in an options YAML file.

You specify the options file in the command using the -b flag.

Here is a basic example of an options file (`options.yaml`) where only the cache directory is set:

In [ ]:
!cat ../examples/options.yaml

This file can be extended to include other Boltz parameters, such as sampling_steps, diffusion_samples and use_potentials.

#### Screening File (CSV)

A **screening file** provides the ligands/peptides/proteins to be evaluated during virtual screening.  

Here’s an example CSV file:

In [ ]:
!cat ../examples/ligand_screen.csv

- `SMILES` is the variable column specified with `--col_variable`.
- `Name` is the compound ID column specified with `--col_id`.
- `pIC50` is an additional property we want to carry forward into the results for post-analysis, so we add it using `--merge_data`.

#### Screening File (SDF)

This same file can also be provided in SDF, which allows the usage of more advanced options using CCD codes. See `advanced tutorial`.

In [ ]:
#TODO: Add example with SDF (exact same as CSV)
#!cat ../examples/ligand_screen.sdf

# Script

In [ ]:
!boltz-tools screen \
    -w ./tmp/screen \
    -s ../examples/system_screen.yaml \
    -b ../examples/options.yaml \
    -v sequences,1,ligand,smiles \
    -c ../examples/ligand_screen.csv \
    --col_variable SMILES \
    --col_id Name \
    --merge_data pIC50

In [8]:
!boltz-tools screen \
    -w ./tmp/screen \
    -s ../examples/system_screen.yaml \
    -b ../examples/options.yaml \
    -v sequences,1,ligand,ccd \
    -c ../examples/ligand_screen.csv \
    --col_variable SMILES \
    --col_id Name \
    --merge_data pIC50 \
    --generate_conformers 3D

INFO:boltz-tools.helpers.ccd:Wrote 4 molecules to ../examples/ligand_screen.sdf
INFO:boltz-tools.helpers.ccd:Generated 3D conformers in ../examples/ligand_screen.sdf
[WARNING] Molecule ID 'erlotinib' is longer than 5 characters. Truncating to 'erlot' to comply with CCD naming rules.

[DEBUG] Checking atom properties just before pickle:
Total atoms: 29
  idx 0 props: {'__computedProps': <rdkit.rdBase._vectSs object at 0x7f1c7c8efbc0>, 'name': 'C1'}
  idx 1 props: {'__computedProps': <rdkit.rdBase._vectSs object at 0x7f1c7c8efbc0>, 'name': 'O2'}
  idx 2 props: {'__computedProps': <rdkit.rdBase._vectSs object at 0x7f1c7c8efbc0>, 'name': 'C3'}
  idx 3 props: {'__computedProps': <rdkit.rdBase._vectSs object at 0x7f1c7c8efbc0>, 'name': 'C4'}
  idx 4 props: {'__computedProps': <rdkit.rdBase._vectSs object at 0x7f1c7c8efbc0>, 'name': 'O5'}
  idx 5 props: {'__computedProps': <rdkit.rdBase._vectSs object at 0x7f1c7c8efbc0>, 'name': 'C6'}
  idx 6 props: {'__computedProps': <rdkit.rdBase._vectSs o

# ---------------------

In [ ]:
import os, pickle
from collections import Counter
from rdkit import Chem
from rdkit.Chem import rdMolInterchange

# ---------- Low-level helpers ----------

def _rehydrate_to_mol(obj):
    """
    Best-effort: turn whatever is in the pickle into an RDKit Mol.
    Handles: Mol, MolBlock string, SMILES string, dict wrappers, etc.
    """
    # Case 1: already an RDKit Mol
    if hasattr(obj, "GetNumAtoms") and callable(obj.GetNumAtoms):
        return obj

    # Case 2: dict wrappers we've seen in the wild
    if isinstance(obj, dict):
        # common keys you might have used earlier
        for k in ("mol", "rdkit_mol", "rdMol", "data"):
            if k in obj:
                maybe = obj[k]
                if hasattr(maybe, "GetNumAtoms"):
                    return maybe
                # might be a MolBlock string or RDKit JSON
                if isinstance(maybe, str):
                    # try MolBlock
                    m = _try_from_text(maybe)
                    if m is not None:
                        return m
        # Could also be a JSON wrapper
        if obj.get("format") == "rdkit-json" and isinstance(obj.get("payload"), str):
            return rdMolInterchange.MolFromJSON(obj["payload"])

    # Case 3: plain string inside the pickle (MolBlock or SMILES)
    if isinstance(obj, str):
        m = _try_from_text(obj)
        if m is not None:
            return m

    raise TypeError("Unrecognized pickle payload; cannot convert to RDKit Mol.")

def _try_from_text(txt):
    """
    Try to interpret text as MolBlock (V2000/V3000) first, then as SMILES.
    """
    # Heuristics to prefer MolBlock first
    looks_like_molblock = ("\nM  END" in txt) or ("V2000" in txt) or ("V3000" in txt)
    if looks_like_molblock:
        m = Chem.MolFromMolBlock(txt, sanitize=False, removeHs=False)
        if m is None:
            # sometimes sanitize=True rescues
            m = Chem.MolFromMolBlock(txt, sanitize=True, removeHs=False)
        if m is not None:
            return m
    # Try SMILES as fallback
    m = Chem.MolFromSmiles(txt)
    return m

def _atom_prop_keys(atom):
    """
    Return all property keys on an atom. We try GetPropNames first,
    then fall back to GetPropsAsDict keys if needed.
    """
    keys = []
    if hasattr(atom, "GetPropNames"):
        try:
            # includePrivate/includeComputed help surface more keys
            keys = list(atom.GetPropNames(includePrivate=True, includeComputed=True))
        except TypeError:
            # older RDKit: no kwargs
            keys = list(atom.GetPropNames())
    if not keys and hasattr(atom, "GetPropsAsDict"):
        try:
            keys = list(atom.GetPropsAsDict(includePrivate=True, includeComputed=True).keys())
        except TypeError:
            keys = list(atom.GetPropsAsDict().keys())
    return keys

# ---------- Forensics / reporting ----------

def inspect_pickle(path):
    with open(path, "rb") as f:
        obj = pickle.load(f)

    print(f"\n=== Inspect: {path} ===")
    print(f"Pickle payload type: {type(obj)}")

    # Try to rehydrate to a Mol
    try:
        mol = _rehydrate_to_mol(obj)
        print("Rehydrated to RDKit Mol ✔")
    except Exception as e:
        print(f"Rehydrate failed: {e}")
        return

    print(f"Atoms: {mol.GetNumAtoms()}, Bonds: {mol.GetNumBonds()}")

    # Gather atom property keys across the whole molecule
    keys_counter = Counter()
    sample_names = []
    for ai, atom in enumerate(mol.GetAtoms()):
        keys = _atom_prop_keys(atom)
        keys_counter.update(keys)
        # record "name-like" if it exists
        for key in ("name","atomName","atomLabel","label","_Name","pdb_name","atom_name","molAtomMapNumber"):
            if atom.HasProp(key):
                sample_names.append((ai, key, atom.GetProp(key)))
                break

    print("Top atom property keys (count across atoms):")
    for k, c in keys_counter.most_common(15):
        print(f"  {k}: {c}")

    if sample_names:
        print("Sample per-atom 'name-like' properties found:")
        for ai, key, val in sample_names[:10]:
            print(f"  atom {ai}: {key} = {val}")
    else:
        print("No obvious 'name-like' atom property found on first pass.")

    return mol

# ---------- Robust loader for your app ----------

def load_cached_mol(path):
    """Unified loader that handles multiple historical cache formats."""
    with open(path, "rb") as f:
        obj = pickle.load(f)
    mol = _rehydrate_to_mol(obj)
    return mol

def get_atom_names(mol, preferred_order=("name","atomName","atomLabel","label","_Name","pdb_name","atom_name")):
    """
    Return a list of per-atom names using the first key present from preferred_order.
    If none are present, returns None.
    """
    # pick the first key that is present on ANY atom
    chosen = None
    for key in preferred_order:
        if any(a.HasProp(key) for a in mol.GetAtoms()):
            chosen = key
            break
    if chosen is None:
        return None, []

    names = []
    for a in mol.GetAtoms():
        if a.HasProp(chosen):
            names.append(a.GetProp(chosen))
        else:
            names.append(None)
    return chosen, names

# ---------- Normalization (optional but recommended) ----------

def normalize_pickle(src_path, dst_path, fmt="rdkit-json"):
    """
    Rewrite a cached molecule into a single consistent format.
    fmt options:
      - 'rdkit-json' : preserves atom properties robustly via rdMolInterchange
      - 'molblock'   : stores a MolBlock string (V2K/V3K)
      - 'pickle-mol' : stores the raw RDKit Mol (may be RDKit-version-sensitive)
    """
    mol = load_cached_mol(src_path)

    if fmt == "rdkit-json":
        payload = {"format": "rdkit-json", "payload": rdMolInterchange.MolToJSON(mol)}
    elif fmt == "molblock":
        payload = Chem.MolToMolBlock(mol)
    elif fmt == "pickle-mol":
        payload = mol
    else:
        raise ValueError("Unknown fmt")

    with open(dst_path, "wb") as f:
        pickle.dump(payload, f)

# ---------- Example usage (what to run) ----------
# 1) Point these to one working and one failing file and run inspect_pickle on both.
inspect_pickle("cache/.boltz/mols/erlotinib.pkl")   # failing
inspect_pickle("cache/.boltz/mols/ZZZ.pkl")         # known-good

# 2) Then, try the unified loader + atom-name probe:
# mol = load_cached_mol("cache/.boltz/mols/erlotinib.pkl")
# key, names = get_atom_names(mol)
# print("Chosen atom-name key:", key)
# print(names)

# 3) Once you know what works, normalize your whole cache in place, e.g.:
# src_dir = "cache/.boltz/mols"
# for fname in os.listdir(src_dir):
#     if not fname.endswith(".pkl"): 
#         continue
#     src = os.path.join(src_dir, fname)
#     tmp = os.path.join(src_dir, fname + ".normalized")
#     try:
#         normalize_pickle(src, tmp, fmt="rdkit-json")
#         os.replace(tmp, src)  # atomically overwrite
#     except Exception as e:
#         print("Normalize failed for", fname, ":", e)


In [ ]:
import pickle
#pkl_file = "cache/.boltz/mols/ZZZ.pkl"
pkl_file = "cache/.boltz/mols/erlotinib.pkl"

with open(pkl_file, "rb") as f:
    mol_loaded = pickle.load(f)


for atom in mol_loaded.GetAtoms():
    print(atom.GetProp("name"))

In [ ]:
pkl_file = "cache/.boltz/mols/erlotinib.pkl"

with open(pkl_file, "rb") as f:
    mol_loaded = pickle.load(f)

# Print general info
print("Number of atoms:", mol.GetNumAtoms())
print("Number of bonds:", mol.GetNumBonds())

# Print atoms
print("\nAtoms:")
for atom in mol.GetAtoms():
    print(f"Index: {atom.GetIdx()}, Symbol: {atom.GetSymbol()}, "
          f"Degree: {atom.GetDegree()}, Formal Charge: {atom.GetFormalCharge()}, "
          f"Hybridization: {atom.GetHybridization()}, Aromatic: {atom.GetIsAromatic()}")

# Print bonds
print("\nBonds:")
for bond in mol.GetBonds():
    print(f"Begin: {bond.GetBeginAtomIdx()}, End: {bond.GetEndAtomIdx()}, "
          f"Type: {bond.GetBondType()}, Is Conjugated: {bond.GetIsConjugated()}, "
          f"Is Aromatic: {bond.GetIsAromatic()}")

# Print properties (if any)
print("\nProperties:")
for prop_name in mol.GetPropNames():
    print(f"{prop_name}: {mol.GetProp(prop_name)}")

# Optional: print SMILES and molecular formula
print("\nSMILES:", Chem.MolToSmiles(mol))
print("Molecular formula:", Chem.rdMolDescriptors.CalcMolFormula(mol))

In [ ]:
import pickle
from rdkit import Chem
from rdkit.Chem import Draw

# Replace with your file path
#pkl_file = "cache/.boltz/mols/ZZZ.pkl"
pkl_file = "cache/.boltz/mols/erlotinib.pkl"

# Load the pickle
with open(pkl_file, "rb") as f:
    data = pickle.load(f)

# Check the type of the object
print("Type of object:", type(data))

# If it's a list or dict, show basic info
if isinstance(data, list):
    print(f"List of length {len(data)}")
    # Show first item
    first_item = data[0]
    print("First item type:", type(first_item))
elif isinstance(data, dict):
    print(f"Dict with keys: {list(data.keys())}")
    first_item = data[list(data.keys())[0]]
    print("Type of first item:", type(first_item))
else:
    first_item = data

# If the first item looks like an RDKit Mol object
if isinstance(first_item, Chem.Mol):
    print("Detected RDKit molecule. Showing molblock:")
    print(Chem.MolToMolBlock(first_item))
    # Optional: display molecule image
    Draw.MolToImage(first_item)
# If it is a string (molblock stored as string)
elif isinstance(first_item, str) and "V2000" in first_item:
    print("Detected possible molblock string:")
    print(first_item[:500])  # show first 500 chars
else:
    print("First item content:")
    print(first_item)


If you want to remove the temporary directory or cache after running the tutorial, uncomment the line below:

In [7]:
!rm -rf ./tmp
#!rm -rf ./cache
!rm -rf ./cache/.boltz/mols/erlotinib.pkl
!rm -rf ./cache/.boltz/mols/erlot.pkl
!rm -rf ./cache/.boltz/mols/gefitinib.pkl
!rm -rf ./cache/.boltz/mols/gefit.pkl

!rm -rf ../examples/tmp/ligand_quick.sdf